# LSSTCam ConstDb visit table and associate tracts and patches

- **Creation date:** 2026-03-26
- **Last update:** 2026-03-26
- **Last update:** 2026-04-07
- **Author:** Sylvie Dagoret-Campagne  
- **Based on:** `2026-03-09_ConsDB_LSSTCam.ipynb`

This notebook queries ConsDB to retrieve LSSTCam visit data (same query as the parent notebook)  

## 0. Imports


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.table import join
from astropy.coordinates import SkyCoord
import astropy.units as u

# Define butler
from lsst.daf.butler import Butler
from lsst.skymap import PatchInfo, Index2D
from lsst.geom import Point2D
from lsst.geom import SpherePoint, degrees
from lsst.summit.utils import ConsDbClient

# %matplotlib widget
%matplotlib inline

## 1. Configuration


In [ ]:
REPO = "main"
COLLECTION = "LSSTCam/defaults"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
DATE_STOP = 20260408
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}" f" and day_obs <= {DATE_STOP}"

TRACT_SEARCH_RADIUS_DEG = 1.8
DDF_RADIUS = TRACT_SEARCH_RADIUS_DEG

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}


print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")


# Visits filename
# FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276_NoTracts.csv"

# FLAGS
FLAG_SAVE_VISITSFILE = True

# Output directory for DRP data
OUTPUT_DIR = "data_DP2_ConstDB_VisitsAndTracts"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. ConsDB connection & data retrieval

Identical queries to the parent notebook `2026-03-09_ConsDB_LSSTCam.ipynb`:  
retrieve all LSSTCam visits since 2025-04-15 and inner-join `visit1` with itself.


In [ ]:
os.environ["no_proxy"] += ",.consdb"
url = "http://consdb-pq.consdb:8080/consdb"
consdb = ConsDbClient(url)
print("ConsDB client created.")

In [ ]:
# Same queries as in the parent notebook
visits = consdb.query("SELECT * FROM cdb_lsstcam.visit1 WHERE day_obs >= 20250415")
visits_ql = consdb.query("SELECT * FROM cdb_lsstcam.visit1")

merged_visits = join(visits, visits_ql, keys="visit_id", join_type="inner")

df_visits = visits.to_pandas()
print(f"Visits retrieved: {len(df_visits)}")

In [ ]:
start_id = df_visits.sort_values(by="visit_id").iloc[0]["visit_id"]
last_id = df_visits.sort_values(by="visit_id").iloc[-1]["visit_id"]

In [ ]:
print(f" first visit {start_id} - last visit {last_id}")

## 3. Visit Data cleaning

Remove non-science entries (engineering filters, pinhole mask) and drop visits  
with missing pointing coordinates.


### 3.1 remove bad filters

In [ ]:
# Remove engineering / pinhole filters (same as parent notebook)
bad_filters = {"other", "none", "other:pinhole", "ph_5"}
df_visits = df_visits[~df_visits["physical_filter"].isin(bad_filters)]

# Remove rows with missing pointing coordinates
df_visits = df_visits.dropna(subset=["s_ra", "s_dec"]).reset_index(drop=True)

print(f"Visits after cleaning : {len(df_visits)}")
print(f'Bands present         : {sorted(df_visits["band"].dropna().unique())}')
print(f'RA  range (deg)       : [{df_visits["s_ra"].min():.2f}, {df_visits["s_ra"].max():.2f}]')
print(f'Dec range (deg)       : [{df_visits["s_dec"].min():.2f}, {df_visits["s_dec"].max():.2f}]')

In [ ]:
df_visits["physical_filter"].unique()

In [ ]:
# df_visits["observation_reason"].unique()

In [ ]:
cut_expos = (df_visits["exp_time"] >= 25) & (df_visits["exp_time"] <= 35)

### 3.2 Selection with time exposure

In [ ]:
cut_expos = (df_visits["exp_time"] >= 25) & (df_visits["exp_time"] <= 35)

In [ ]:
df_visits = df_visits[cut_expos].copy()

## 4. Butler access 

In [ ]:
def get_unique_tract_centers(df, skymap, raname="ra", decname="dec"):
    tract_dict = {}

    for _, row in df.iterrows():
        if pd.isna(row[raname]) or pd.isna(row[decname]):
            continue

        sp = SpherePoint(row[raname], row[decname], degrees)
        tract = skymap.findTract(sp)
        tract_id = tract.getId()

        # éviter les doublons
        if tract_id not in tract_dict:
            center = tract.getCtrCoord()
            tract_dict[tract_id] = {"ra": center.getRa().asDegrees(), "dec": center.getDec().asDegrees()}

    return tract_dict

### 4.1 Butler

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

### 4.2 SkyMap

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception as inst:
    print(type(inst))  # the exception type
    print(inst.args)  # arguments stored in .args
    print(inst)  # __str__ allows args to be printed directly,
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)

print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")
# skymap = butler.get("skyMap", skymap=skymapName, collections=collection_validation)

## 5. Associate tract and patch to visits

In [ ]:
def get_tract_patch(row, skymap):
    if pd.isna(row["s_ra"]) or pd.isna(row["s_dec"]):
        return pd.Series({"tract": None, "patch": None})

    target_point = SpherePoint(row["s_ra"], row["s_dec"], degrees)

    tract_info = skymap.findTract(target_point)
    patch_info = tract_info.findPatch(target_point)
    tractNbSel = tract_info.getId()
    patchNbSel = patch_info.getSequentialIndex()
    patch_index_str = f"{patch_info.getIndex()[0]},{patch_info.getIndex()[1]}"
    return pd.Series({"tract": tractNbSel, "patch": patchNbSel, "patch_str": patch_index_str})

In [ ]:
def get_tract_patch(row, skymap, raname="ra", decname="dec"):
    if pd.isna(row[raname]) or pd.isna(row[decname]):
        return pd.Series(
            {
                "tract": None,
                "patch": None,
                "patch_str": None,
                "tract_bbox": None,
                "tract_ra_corners": None,
                "tract_dec_corners": None,
            }
        )

    target_point = SpherePoint(row[raname], row[decname], degrees)

    tract_info = skymap.findTract(target_point)
    patch_info = tract_info.findPatch(target_point)

    # IDs
    tractNbSel = tract_info.getId()
    patchNbSel = patch_info.getSequentialIndex()
    patch_index_str = f"{patch_info.getIndex()[0]},{patch_info.getIndex()[1]}"

    # =========================
    # BBOX en pixels
    # =========================
    bbox = tract_info.getBBox()
    bbox_tuple = (bbox.getMinX(), bbox.getMinY(), bbox.getMaxX(), bbox.getMaxY())

    # =========================
    # Corners in sky coordinates (RA/Dec)
    # =========================
    wcs = tract_info.getWcs()
    corners = bbox.getCorners()

    ra_list = []
    dec_list = []

    for corner in corners:
        corner2d = Point2D(corner)
        sky = wcs.pixelToSky(corner2d)
        ra_list.append(sky.getRa().asDegrees())
        dec_list.append(sky.getDec().asDegrees())

    return pd.Series(
        {
            "tract": tractNbSel,
            "patch": patchNbSel,
            "patch_str": patch_index_str,
            "tract_bbox": bbox_tuple,
            "tract_ra_corners": ra_list,
            "tract_dec_corners": dec_list,
        }
    )

In [ ]:
# list(df_visits.columns)

In [ ]:
df = df_visits.copy()

In [ ]:
# compute tract-patch
# df_tractpatch = df.apply(get_tract_patch, axis=1, args=(skymap,raname = "s_ra",decname = "s_dec"))

df_tractpatch = df.apply(lambda row: get_tract_patch(row, skymap, raname="s_ra", decname="s_dec"), axis=1)

# df_tractpatch = df.apply(
#    get_tract_patch,
#    axis=1,
#    args=(skymap, "s_ra", "s_dec")
# )

In [ ]:
df = pd.concat([df, df_tractpatch], axis=1)
df["tract"] = df["tract"].astype("Int64")
df["patch"] = df["patch"].astype("Int64")

In [ ]:
df.head()

In [ ]:
df.tail()

## Save in a parquet file

In [ ]:
id_min = df.visit_id.min()
id_max = df.visit_id.max()
nvisits = len(df)

In [ ]:
print(f"\t >> ConstDb visits : {nvisits} visits ==> first visit = {id_min}, last visit = {id_max}")

In [ ]:
if FLAG_SAVE_VISITSFILE:
    output_filename_parquet = f"constDbVisitTable-{id_min}-{id_max}_N{nvisits}_WithTracts.parquet"
    output_fullfilename_parquet = os.path.join(OUTPUT_DIR, output_filename_parquet)
    df.to_parquet(output_fullfilename_parquet)

In [ ]:
print(output_fullfilename_parquet)